# VLSI Timing Dataset — Exploration & Visualization

EDA for the ML-assisted standard-cell timing project. Run the pipeline first:

```
python run_all.py --step data
```

This notebook verifies the physics, visualizes delay vs PVT, and sanity-checks the ML model if trained.

In [ ]:
import sys, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == 'notebooks') else pathlib.Path.cwd()
sns.set_theme(style='darkgrid')
df = pd.read_csv(ROOT / 'data' / 'processed' / 'timing_dataset.csv')
print(df.shape)
df.head()

## 1. Delay distribution per cell (log scale)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for cell, g in df.groupby('cell'):
    sns.kdeplot(g['tpd_ps'], ax=ax[0], label=cell, fill=True, alpha=0.3)
    sns.kdeplot(np.log(g['tpd_ps']), ax=ax[1], label=cell, fill=True, alpha=0.3)
ax[0].set_title('tpd (ps)'); ax[1].set_title('log(tpd) — more Gaussian')
ax[0].legend(); ax[1].legend(); plt.tight_layout()

## 2. Physics check — monotonic relationships
Delay should drop with VDD, rise with temperature, SS slowest / FF fastest, and halve with drive.

In [ ]:
inv = df[(df.cell=='inv') & (df.cload_ff.between(9,11)) & (df.drive_strength==2)]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.lineplot(data=inv, x='vdd', y='tpd_ps', hue='proc', ax=ax[0], errorbar='sd')
ax[0].set_title('tpd vs VDD (should decrease)')
sns.lineplot(data=inv, x='temp', y='tpd_ps', hue='proc', ax=ax[1], errorbar='sd')
ax[1].set_title('tpd vs Temp (should increase)')
d = df[(df.cell=='inv') & (df.proc=='tt')]
sns.lineplot(data=d, x='drive_strength', y='tpd_ps', ax=ax[2], errorbar='sd')
ax[2].set_title('tpd vs drive (≈ 1/drive)'); plt.tight_layout()

## 3. Corner ordering — SS slowest, FF fastest

In [ ]:
order = ['ff','sf','tt','fs','ss']
sns.boxplot(data=df[df.cell=='inv'], x='proc', y='tpd_ps', order=order)
plt.title('INV delay by process corner'); plt.show()
print(df[df.cell=='inv'].groupby('proc')['tpd_ps'].mean().sort_values())

## 4. Feature correlation

In [ ]:
feat = ['vdd','temp','cload_ff','drive_strength','proc_factor','vdd_minus_vth','tpd_ps']
sns.heatmap(df[feat].corr(), annot=True, fmt='.2f', cmap='magma'); plt.title('correlations'); plt.show()

## 5. (optional) ML model sanity check

In [ ]:
sys.path.insert(0, str(ROOT / 'ml'))
try:
    from inference import TimingInference
    eng = TimingInference()
    print(eng.predict(cell='inv', vdd=1.8, temp=27, cload_ff=10, drive=2, corner='tt'))
    tbl = eng.predict_corner_table('inv', 2, 10.0)
    piv = tbl.pivot_table(index=['vdd','temp'], columns='corner', values='tpd_ps')
    display(piv.round(2))
except Exception as e:
    print('Train the model first (python run_all.py --step train). ', e)